# convMMD deconvolution and density estimation

This package-backed example estimates a one-dimensional latent density from additive Laplace-corrupted observations with known per-observation error scales. The latent values generated below are retained only for evaluation; fitting receives only the noisy observations and known error standard deviations.

Set `CONVMMD_NOTEBOOK_SMOKE=1` before execution to use the reduced release-check configuration. The default remains a single illustrative run rather than a paper reproduction or performance claim.


In [ ]:
import os

import matplotlib.pyplot as plt

import numpy as np

import torch

from convMMD.core.data import (

    generate_1d_laplace_mixture,

    true_density_1d_laplace_mixture,

)

from convMMD.core.evaluate import compute_ise

from convMMD.density_models import NormalizingFlowDensity

from convMMD.training import train_convmmd


## 1. Fixed configuration and latent simulation


In [ ]:
SMOKE = os.getenv("CONVMMD_NOTEBOOK_SMOKE") == "1"

SEED = 20260829

DEVICE = "cuda" if torch.cuda.is_available() and not SMOKE else "cpu"

N_SAMPLES = 64 if SMOKE else 1000

EPOCHS = 2 if SMOKE else 500

BATCH_SIZE = 32 if SMOKE else 256

MODEL_KWARGS = {

    "num_blocks": 1 if SMOKE else 4,

    "num_bins": 4 if SMOKE else 16,

    "hidden_features": 8 if SMOKE else 32,

    "tail_bound": 30.0,

}

latent_truth, noisy_observations, known_noise_std = generate_1d_laplace_mixture(

    n_samples=N_SAMPLES,

    noise_type="laplace",

    seed=SEED,

    device=DEVICE,

)

print(f"device={DEVICE}, smoke={SMOKE}, n={N_SAMPLES}, epochs={EPOCHS}")


## 2. Fit from noisy observations

This cell contains the entire fitting path. `latent_truth` is deliberately absent: the flow is trained by matching noisy observations to forward-convolved latent samples through the public package API.


In [ ]:
torch.manual_seed(SEED + 1)

model = NormalizingFlowDensity(dim=1, flow_type="nsf", **MODEL_KWARGS)

fit = train_convmmd(

    model=model,

    x_noisy=noisy_observations,

    noise_std=known_noise_std,

    noise_type="laplace",

    kernel_type="laplace",

    epochs=EPOCHS,

    batch_size=BATCH_SIZE,

    warmup_epochs=0 if SMOKE else None,

    bandwidths=[0.5, 1.0, 2.0] if SMOKE else None,

    eval_every=max(1, EPOCHS // 5),

    device=DEVICE,

    verbose=not SMOKE,

)

fitted_model = fit["model"]


## 3. Simulation-only evaluation


In [ ]:
ise = compute_ise(

    fitted_model,

    true_density_1d_laplace_mixture,

    device=DEVICE,

    resolution=256 if SMOKE else 2000,

)

print(f"Integrated squared error against latent truth: {ise:.6f}")

grid = torch.linspace(-8.0, 10.0, 400, device=DEVICE)[:, None]

with torch.no_grad():

    fitted_density = torch.exp(fitted_model.log_prob(grid)).cpu().numpy()

grid_np = grid[:, 0].cpu().numpy()

true_density = true_density_1d_laplace_mixture(grid_np)

fig, ax = plt.subplots(figsize=(8, 4.5))

ax.hist(

    noisy_observations[:, 0].cpu().numpy(),

    bins=35,

    density=True,

    alpha=0.25,

    color="tab:gray",

    label="Noisy observations",

)

ax.plot(grid_np, true_density, linewidth=2, label="Latent truth")

ax.plot(grid_np, fitted_density, linewidth=2, label="convMMD estimate")

ax.set(xlabel="Value", ylabel="Density", title="Latent-density deconvolution")

ax.grid(alpha=0.2)

ax.legend()

plt.tight_layout()

plt.show()


## Interpretation

ISE is available only because this notebook generated the latent distribution. For real data, use observed-space forward checks and sensitivity to seeds, model size, and training budget. The reduced smoke mode verifies execution only.
